# AI-Powered Customer Churn Intelligence System

## Day 6 - LLM-Powered Retention Recommendations

This notebook uses customer-level churn predictions and SHAP explanations
to generate personalized retention recommendations using a Large Language
Model accessed through OpenRouter.

Workflow:

Customer Profile
        ↓
Churn Probability
        ↓
SHAP Explanation
        ↓
Customer Risk Context
        ↓
LLM
        ↓
Personalized Retention Strategy

In [1]:
import os
import json
import requests
import pandas as pd

from dotenv import load_dotenv

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
model_name = os.getenv("OPENROUTER_MODEL")

print("API key loaded:", bool(api_key))
print("Model:", model_name)

API key loaded: True
Model: openrouter/free


In [5]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
model_name = os.getenv("OPENROUTER_MODEL")

print("API key exists:", bool(api_key))
print("API key length:", len(api_key) if api_key else 0)
print("API key prefix:", api_key[:8] if api_key else "None")
print("Model:", model_name)

API key exists: True
API key length: 73
API key prefix: sk-or-v1
Model: openrouter/free


In [6]:
import requests

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

payload = {
    "model": model_name,
    "messages": [
        {
            "role": "user",
            "content": "Reply with exactly: OpenRouter connection successful."
        }
    ],
    "temperature": 0
}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=60
)

print("Status code:", response.status_code)
print("Response:", response.text[:1000])

Status code: 200
Response: 
         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         
{"id":"gen-1786362690-gb1I2g4PvixizvUjrAq3","object":"chat.completion","created":1786362690,"model":"nvidia/nemotron-3-ultra-550b-a55b:free","provider":"Nvidia","system_fingerprint":null,"service_tier":null,"choices":[{"index":0,"logprobs":null,"finish_reason":"stop","native_finish_reason":"stop","message":{"role":"assistant","content":"OpenRouter connection successful.","refusal":null,"reasoning":

In [7]:
print("Status code:", response.status_code)
print("Content length:", len(response.text))
print("Raw response:", repr(response.text))

if response.text.strip():
    result = response.json()

    print("\nModel used:", result.get("model"))

    if "choices" in result and result["choices"]:
        print("\nLLM response:")
        print(result["choices"][0]["message"]["content"])
    else:
        print("\nNo choices returned.")
else:
    print("\nOpenRouter returned an empty response.")

Status code: 200
Content length: 1734
Raw response: '\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n{"id":"gen-1786362690-gb1I2g4PvixizvUjrAq3","object":"chat.completion","created":1786362690,"model":"nvidia/nemotron-3-ultra-550b-a55b:free","provider":"Nvidia","system_fingerprint":null,"service_tier":null,"choices":[{"index":0,"logprobs":null,"finish_reason":"stop","na

In [8]:
customer_context = """
Customer Churn Analysis

Customer ID: 2461
Predicted Churn Probability: 95.87%

Customer Profile:
- Subscription Plan: Basic
- Monthly Fee: 199
- Average Weekly Usage: 3.7 hours
- Support Tickets: 8
- Payment Failures: 5
- Tenure: 1 month
- Days Since Last Login: 45 days
- Login Recency: Inactive
- Support Risk: High
- Payment Risk: High
- Usage Level: Low

Top SHAP Drivers:
1. Low usage level: SHAP +1.2683
2. Support tickets: SHAP +0.9662
3. Payment failures: SHAP +0.5876
4. Inactive login: SHAP +0.3244
5. Days since last login: SHAP +0.1820

Negative SHAP factors:
- Low support risk: SHAP -0.2537
- Average weekly usage hours: SHAP -0.2224
"""

print(customer_context)


Customer Churn Analysis

Customer ID: 2461
Predicted Churn Probability: 95.87%

Customer Profile:
- Subscription Plan: Basic
- Monthly Fee: 199
- Average Weekly Usage: 3.7 hours
- Support Tickets: 8
- Payment Failures: 5
- Tenure: 1 month
- Days Since Last Login: 45 days
- Login Recency: Inactive
- Support Risk: High
- Payment Risk: High
- Usage Level: Low

Top SHAP Drivers:
1. Low usage level: SHAP +1.2683
2. Support tickets: SHAP +0.9662
3. Payment failures: SHAP +0.5876
4. Inactive login: SHAP +0.3244
5. Days since last login: SHAP +0.1820

Negative SHAP factors:
- Low support risk: SHAP -0.2537
- Average weekly usage hours: SHAP -0.2224



In [13]:
import requests

prompt = f"""
You are a customer retention analyst for a subscription business.

Analyze the following customer churn information and create an actionable
retention strategy.

{customer_context}

Your response must contain exactly these sections:

1. CHURN ASSESSMENT
- State the risk level based on the predicted churn probability.

2. KEY CHURN REASONS
- Identify the 3 most important reasons for the customer's churn risk.
- Use the SHAP values to support the reasoning.

3. RETENTION ACTIONS
- Give 3 specific actions the business should take.
- Prioritize actions that directly address the customer's risk factors.

4. PRIORITY
- Classify the customer as Critical, High, Medium, or Low priority.

5. BUSINESS EXPLANATION
- Give a short explanation that a non-technical business manager can understand.

Do not invent customer information.
Do not assume that support tickets are open or unresolved unless explicitly stated.
Do not claim that SHAP proves causation.
Do not describe the customer as certain or guaranteed to churn; describe the result as a predicted likelihood.
Keep the recommendation practical and concise.
"""

payload = {
    "model": model_name,
    "messages": [
        {
            "role": "system",
            "content": "You are an expert customer retention and business analytics assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    "temperature": 0.2
}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=60
)

print("Status code:", response.status_code)

response.raise_for_status()

result = response.json()

llm_recommendation = result["choices"][0]["message"]["content"]

print(llm_recommendation)

Status code: 200
1. CHURN ASSESSMENT
- Risk Level: Critical. With a predicted churn probability of 95.87%, this customer is at extremely high risk of churning.

2. KEY CHURN REASONS
- Low usage level (SHAP +1.2683): The customer's usage is significantly below average, indicating low engagement with the service.
- High number of support tickets (SHAP +0.9662): The customer has submitted 8 support tickets, suggesting ongoing issues or dissatisfaction.
- Payment failures (SHAP +0.5876): With 5 payment failures, the customer is experiencing difficulties with the payment process, which is a strong indicator of potential churn.

3. RETENTION ACTIONS
- Proactively reach out to the customer with a personalized call or email to address their support concerns and offer assistance with any unresolved issues.
- Offer a payment method review and assistance, potentially providing alternative payment options or a one-time payment adjustment to resolve the payment failures.
- Provide a targeted engage

In [14]:
import json
from pathlib import Path

recommendation_data = {
    "customer_id": 2461,
    "predicted_churn_probability": 0.9587,
    "model": model_name,
    "recommendation": llm_recommendation
}

output_path = Path("../models/customer_2461_llm_recommendation.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(recommendation_data, f, indent=4, ensure_ascii=False)

print(f"Saved recommendation to: {output_path}")

Saved recommendation to: ..\models\customer_2461_llm_recommendation.json
